# Data Analysis Notebook

### **This is the new notebook, which includes the logic for the baseline. Instead of Total Energy, we are in general taking into consideration**
To isolate the energy consumption strictly attributable to the Large Language Models, we executed a control group of 30 idle trials. By measuring the system's baseline power draw during these sleep states, we established a constant idle threshold of [X] Watts. This allows us to differentiate between the static hardware overhead and the dynamic energy ($\Delta E$) actively consumed by the AI inference algorithms, ensuring a highly accurate representation of software-level sustainability.

Dynamic Energy = Total_Energy - (Idle_Power * Time)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import glob
import os
import scipy.stats as stats
import re

In [ ]:
# Define data directory
base_dir = "../results/experiment-002"
# Find all folders (models + control)
model_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]

## Trapezoid Rule for Total Energy

We analyze each csv for each model. We use the Trapezoid rule to calculate the Total Energy (Joules) for a specific trial and we repeat 30 Times: Do this for all 30 CSVs in that model's folder. 

In [ ]:
# We will store every single trial's final result in this list
all_trials_data = []

print("Processing all CSV files using the Trapezoid Rule...")

for folder in model_folders:
    # Find all 30 CSVs in this folder
    csv_pattern = os.path.join(base_dir, folder, "trial-*.csv")
    csv_files = glob.glob(csv_pattern)
    
    for file in csv_files:
        df = pd.read_csv(file)
        
        # Failsafe: check if necessary columns exist
        if 'GPU0_POWER (mWatts)' not in df.columns or 'GPU0_TEMPERATURE' not in df.columns:
            continue
            
        # Convert timestamp to elapsed Seconds (starting from 0)
        time_sec = (df['Time'] - df['Time'].min()) / 1000.0
        
        # Convert milliWatts to standard Watts
        power_watts = df['GPU0_POWER (mWatts)'] / 1000.0
        
        # Calculate Area Under the Curve (Total Joules)
        total_energy_joules = np.trapezoid(y=power_watts, x=time_sec)
        
        # Get the total duration
        total_time_seconds = time_sec.max()
        
        # Extract Temperature Metrics
        max_temp = df['GPU0_TEMPERATURE'].max()
        start_temp = df['GPU0_TEMPERATURE'].iloc[0]
        
        # Store the calculated metrics for THIS trial
        all_trials_data.append({
            'Model': folder,
            'Trial_Name': os.path.basename(file),
            'Total_Energy_Joules': total_energy_joules,
            'Total_Time_Seconds': total_time_seconds,
            'Max_Temp': max_temp,      # Add this
            'Start_Temp': start_temp   # Add this
        })

# Convert the master list into a DataFrame
df_results = pd.DataFrame(all_trials_data)

print(f"\nSuccessfully processed {len(df_results)} trials.")
print("\nHere is a sneak peek of the dataset:")
display(df_results.head())

## Anomalies Detection - Data Cleaning

Running the following block of code, we identified some executions were anomalous. Few trials reached our 60-second hardware timeout limit and were forcefully aborted, resulting in extreme energy spikes and for some the model failed to load. Following standard data-cleaning procedures, these outliers were discarded to prevent skewed distributions.

In [ ]:
print("=== IDENTIFYING FAILED GENERATIONS FROM LOGS ===")
file_path = "../results/experiment-002/responses.txt"
with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
    raw_text = f.read()

# Split the text file into individual trial blocks
blocks = re.split(r'===\s+Trial\s+([0-9]+)\s+-\s+Model:\s+([^\s]+)\s+.*?\s+===', raw_text)

invalid_trial_keys = set() 

for i in range(1, len(blocks), 3):
    trial_str = f"trial-{int(blocks[i]):03d}.csv"
    model_raw = blocks[i+1] 
    
    # Just replace the colon with an underscore to match your folder names
    model_name = model_raw.replace(':', '_')
        
    text_content = blocks[i+2].strip().lower() 
    
    # EXACT MATCH LOGIC
    if "failed to load model" in text_content:
        invalid_trial_keys.add(f"{model_name}|{trial_str}")

print(f"Identified {len(invalid_trial_keys)} failed/broken executions from text logs:")
for key in sorted(invalid_trial_keys):
    print(f"  -> Flagged: {key}")
print("\n=== OUTLIER & ERROR DETECTION REPORT ===")

valid_trials = []

# Flag and filter outliers per model
for model in df_results['Model'].unique():
    model_df = df_results[df_results['Model'] == model].copy()
    
    # Create a matching key to check against our text-log blacklist
    model_df['Run_Key'] = model_df['Model'] + "|" + model_df['Trial_Name']
    
    # Calculate Mean and Standard Deviation for Time
    mean_time = model_df['Total_Time_Seconds'].mean()
    std_time = model_df['Total_Time_Seconds'].std()
    
    if model == 'control':
        outlier_condition = (
            (model_df['Total_Time_Seconds'] < 1.0) | 
            (np.abs(model_df['Total_Time_Seconds'] - mean_time) > 3 * std_time)
        )
    else:
        # Filters crashes, timeouts, 3x StdDev, and the text file blacklist
        outlier_condition = (
            #(model_df['Total_Time_Seconds'] < 1.0) | 
            (model_df['Total_Time_Seconds'] >= 59.0) |
            (np.abs(model_df['Total_Time_Seconds'] - mean_time) > 3 * std_time) |
            (model_df['Run_Key'].isin(invalid_trial_keys))
        )
    
    outliers = model_df[outlier_condition]
    clean_data = model_df[~outlier_condition]
    
    # Drop the temporary Run_Key column before saving
    clean_data = clean_data.drop(columns=['Run_Key'])
    valid_trials.append(clean_data)
    
    if not outliers.empty: #and model != 'control':
        print(f"⚠️ {model} had {len(outliers)} anomalous or failed runs discarded:")
        for _, row in outliers.iterrows():
            exact_key = f"{model}|{row['Trial_Name']}"
            
            # This determines why the file was dropped
            if exact_key in invalid_trial_keys:
                reason = "Failed to load model"
            elif row['Total_Time_Seconds'] >= 59.0:
                reason = "Timeout (>= 59s)"
            elif row['Total_Time_Seconds'] < 1.0:
                reason = "Immediate Crash (< 1s)"
            else:
                reason = "Statistical Time Anomaly (> 3 StdDev)"
                
            print(f"   - {row['Trial_Name']}: Time = {row['Total_Time_Seconds']:.2f}s | Reason: {reason}")
        print()

# Create the final, clean dataset
clean_df = pd.concat(valid_trials)

print(f"Data cleaning complete. Retained {len(clean_df)} valid trials out of {len(df_results)} total.")

### Calculating Global Dynamic Energy, System's Baseline Idle Power

In [ ]:
print("=== CALCULATING GLOBAL DYNAMIC ENERGY ===")

# Calculate the System's Baseline Idle Power (Watts)
control_df = clean_df[clean_df['Model'] == 'control']
idle_power_watts = (control_df['Total_Energy_Joules'] / control_df['Total_Time_Seconds']).mean()

print(f"System Baseline Idle Power: {idle_power_watts:.2f} Watts")

# Convert all Total Energy into Dynamic Energy (ΔE)
clean_df['Dynamic_Energy_Joules'] = clean_df['Total_Energy_Joules'] - (idle_power_watts * clean_df['Total_Time_Seconds'])

# Optional: Prevent negative energy on weird micro-fluctuations in idle runs
clean_df['Dynamic_Energy_Joules'] = clean_df['Dynamic_Energy_Joules'].clip(lower=0)

print("✅ 'Dynamic_Energy_Joules' successfully added to the dataset!")

### Checking Temperature

In [ ]:
# Get the peak temperature per model directly from our clean DataFrame
# This finds the single highest temperature reached in any trial for each model
max_temps_series = clean_df.groupby('Model')['Max_Temp'].max()

# Identify the absolute global maximum recorded across the whole experiment
global_max = max_temps_series.max()

# Print the final conclusive statement
print(f"\nThe absolute Maximum GPU Temperature recorded was {global_max}°C.")
if global_max <= 66:
    print("Conclusion Validated: Thermal throttling was successfully prevented.")
else:
    print(f"Note: Peak temperature reached {global_max}°C. Still likely below throttling limits.")

# Visual proof for the report
plt.figure(figsize=(12, 6))

# Shorten the names for the X-axis labels to match previous plots
short_names = [name.replace('deepseek-r1_7b-qwen-distill-', 'DS-').replace('qwen2.5_7b-instruct-', 'Qwen-').upper() 
               for name in max_temps_series.index]

bars = plt.bar(short_names, max_temps_series.values, color='#e74c3c', edgecolor='black', alpha=0.8)

# Add the 85°C safety line (standard for NVIDIA/modern GPUs)
plt.axhline(y=85, color='red', linestyle='--', linewidth=2, label='Typical Thermal Throttling Limit (85°C)')

plt.title("Peak GPU Temperature per AI Model Configuration", fontsize=15, fontweight='bold')
plt.ylabel("Maximum Temperature reached (°C)", fontsize=12)
plt.xticks(rotation=35, ha='right', fontsize=10)

plt.ylim(0, 100) 
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(loc='upper left')

# Add the exact temperature number on top of each bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1.5, f"{int(yval)}°C", 
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/experiment-002/temperature-boxplot.png', dpi=300)
plt.show()

## Energy Consumption

### Baseline Subtraction (control run)

To isolate the energy consumption strictly attributable to the Large Language Models, we executed a control group of 30 idle trials. By measuring the system's baseline power draw during these sleep states, we established a constant idle threshold of [X] Watts. This allows us to differentiate between the static hardware overhead and the dynamic energy ($\Delta E$) actively consumed by the AI inference algorithms, ensuring a highly accurate representation of software-level sustainability.

In [ ]:
# Prepare data for plotting (exclude control from the plot itself)
plot_df = clean_df[clean_df['Model'] != 'control'].copy()

# Shorten the names to make the X-axis readable
plot_df['Model_Short'] = plot_df['Model'].str.replace('deepseek-r1_7b-qwen-distill-', 'DS-Distilled-')
plot_df['Model_Short'] = plot_df['Model_Short'].str.replace('qwen2.5_7b-instruct-', 'Qwen-')

# We group 4-bit together, then 8-bit, then 16-bit
quantization_order = [
    'Qwen-q4_K_M', 'DS-Distilled-q4_K_M', 
    'Qwen-q8_0',   'DS-Distilled-q8_0', 
    'Qwen-fp16',   'DS-Distilled-fp16'
]

# Draw the Plot
plt.figure(figsize=(14, 8))
sns.set_theme(style="whitegrid")

# Create the Violin Plot using our new DYNAMIC ENERGY
sns.violinplot(
    x='Model_Short', 
    y='Dynamic_Energy_Joules', 
    data=plot_df, 
    order=quantization_order,
    inner=None, 
    color="lightgray", 
    linewidth=1.5,
    alpha=0.6
)

# Overlay the Box Plot (also using DYNAMIC ENERGY)
sns.boxplot(
    x='Model_Short', 
    y='Dynamic_Energy_Joules', 
    data=plot_df, 
    order=quantization_order, 
    width=0.2, 
    boxprops={'zorder': 2, 'alpha': 0.9},
    palette="Set2" 
)

plt.title("Energy Consumption Distribution by Model & Precision", fontsize=16, fontweight='bold')
plt.xlabel("Model Version (4-bit, 8-bit, 16-bit)", fontsize=12)
plt.ylabel("Total Energy (Joules)", fontsize=12)

plt.xticks(rotation=20, ha='right', fontsize=11)

plt.tight_layout()
plt.savefig('../images/experiment-002/energy_violin_boxplot.png', dpi=300)
plt.show()

### Shapiro - Wilk test & T-test for Normal or Mann-Whitney U Test for not Normal

In [ ]:
# Define the pairs we want to compare in a list
model_pairs = [
    ('qwen2.5_7b-instruct-q4_K_M', 'deepseek-r1_7b-qwen-distill-q4_K_M', '4-bit'),
    ('qwen2.5_7b-instruct-q8_0', 'deepseek-r1_7b-qwen-distill-q8_0', '8-bit'),
    ('qwen2.5_7b-instruct-fp16', 'deepseek-r1_7b-qwen-distill-fp16', '16-bit')
]

for qwen_name, ds_name, precision in model_pairs:
    print(f"======================================================")
    print(f"==== STATISTICAL SIGNIFICANCE TEST: {precision.upper()} MODELS ====")
    print(f"======================================================\n")
    
    # Isolate the data for the current pair
    qwen_data = clean_df[clean_df['Model'] == qwen_name]['Dynamic_Energy_Joules']
    ds_data = clean_df[clean_df['Model'] == ds_name]['Dynamic_Energy_Joules']
    
    # The Normality Test (Shapiro-Wilk)
    stat_q, p_q = stats.shapiro(qwen_data)
    stat_d, p_d = stats.shapiro(ds_data)
    
    print(f"Shapiro-Wilk p-value (Qwen): {p_q:.5f}")
    print(f"Shapiro-Wilk p-value (DeepSeek): {p_d:.5f}")
    
    # Choose the right test dynamically
    if p_q > 0.05 and p_d > 0.05:
        print("-> Both p-values are > 0.05. The data IS normally distributed.")
        print("-> Action: Using the Parametric Welch's T-Test.\n")
        stat_val, p_value = stats.ttest_ind(qwen_data, ds_data, equal_var=False)
        test_name = "Welch's T-Test"
        
        # Use Means for the Effect Size since data is normal
        val_qwen = qwen_data.mean()
        val_ds = ds_data.mean()
        metric_name = "Mean"
    else:
        print("-> At least one p-value is < 0.05. The data is NOT normally distributed.")
        print("-> Action: Using the Non-Parametric Mann-Whitney U Test.\n")
        stat_val, p_value = stats.mannwhitneyu(qwen_data, ds_data, alternative='two-sided')
        test_name = "Mann-Whitney U Test"
        
        # Use Medians for the Effect Size since data is heavily skewed
        val_qwen = qwen_data.median()
        val_ds = ds_data.median()
        metric_name = "Median"
        
    # Print Final Significance
    print(f"{test_name} p-value: {p_value:.5e}")
    if p_value < 0.05:
        print(f"-> CONCLUSION: The energy difference between the {precision} models is STATISTICALLY SIGNIFICANT.\n")
    else:
        print(f"-> CONCLUSION: The difference is not statistically significant (just random noise).\n")

### Effect Size - Horizontal Comparison (Qwen vs Deepseek) and Vertical Comparison (Quantization Savings)

In [ ]:
print("===========================================================")
print("=== TABLE 1: HORIZONTAL EFFECT SIZE (QWEN VS DEEPSEEK) ====")
print("===========================================================\n")

model_pairs = [
    ('qwen2.5_7b-instruct-q4_K_M', 'deepseek-r1_7b-qwen-distill-q4_K_M', '4-bit'),
    ('qwen2.5_7b-instruct-q8_0', 'deepseek-r1_7b-qwen-distill-q8_0', '8-bit'),
    ('qwen2.5_7b-instruct-fp16', 'deepseek-r1_7b-qwen-distill-fp16', '16-bit')
]

horizontal_results = []

for qwen_name, ds_name, precision in model_pairs:
    qwen_data = clean_df[clean_df['Model'] == qwen_name]['Dynamic_Energy_Joules']
    ds_data = clean_df[clean_df['Model'] == ds_name]['Dynamic_Energy_Joules']
    
    # Check normality again to ensure we use the mathematically correct metric
    _, p_q = stats.shapiro(qwen_data)
    _, p_d = stats.shapiro(ds_data)
    
    # IF NORMAL: Use Mean & Cohen's d
    if p_q > 0.05 and p_d > 0.05:
        val_qwen, val_ds = qwen_data.mean(), ds_data.mean()
        metric_used = "Mean"
        
        # Calculate Cohen's d
        n1, n2 = len(qwen_data), len(ds_data)
        s1, s2 = np.var(qwen_data, ddof=1), np.var(ds_data, ddof=1)
        s_pooled = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))
        
        es_value = (val_qwen - val_ds) / s_pooled
        es_name = "Cohen's d"
        
    # IF NON-NORMAL: Use Median & Vargha-Delaney A12
    else:
        val_qwen, val_ds = qwen_data.median(), ds_data.median()
        metric_used = "Median"
        
        # Calculate Vargha-Delaney A12 using Mann-Whitney U
        u_stat, _ = stats.mannwhitneyu(qwen_data, ds_data, alternative='two-sided')
        
        es_value = u_stat / (len(qwen_data) * len(ds_data))
        es_name = "A12 (Vargha-Delaney)"
        
    # Calculate Percentage Savings
    savings = ((val_qwen - val_ds) / val_qwen) * 100
    
    horizontal_results.append({
        'Precision': precision.upper(),
        'Center Metric': metric_used,
        'Qwen Energy (J)': round(val_qwen, 1),
        'DeepSeek Energy (J)': round(val_ds, 1),
        'Savings (Qwen->DS)': f"{savings:.1f}%",
        'Effect Size Metric': es_name,            
        'Effect Size Value': round(es_value, 2)   
    })

# Print the horizontal table cleanly
es_horizontal_df = pd.DataFrame(horizontal_results)
print(es_horizontal_df.to_string(index=False))
print("\n\n")

print("============================================================")
print("=== TABLE 2: VERTICAL EFFECT SIZE (QUANTIZATION SAVINGS) ===")
print("============================================================\n")

families = {
    'Qwen 2.5 (7B)': {
        'fp16': 'qwen2.5_7b-instruct-fp16',
        'q8': 'qwen2.5_7b-instruct-q8_0',
        'q4': 'qwen2.5_7b-instruct-q4_K_M'
    },
    'DeepSeek R1 (7B)': {
        'fp16': 'deepseek-r1_7b-qwen-distill-fp16',
        'q8': 'deepseek-r1_7b-qwen-distill-q8_0',
        'q4': 'deepseek-r1_7b-qwen-distill-q4_K_M'
    }
}

vertical_results = []

for family_name, models in families.items():
    # Extract the raw data arrays for statistical testing
    data_fp16 = clean_df[clean_df['Model'] == models['fp16']]['Dynamic_Energy_Joules']
    data_q8   = clean_df[clean_df['Model'] == models['q8']]['Dynamic_Energy_Joules']
    data_q4   = clean_df[clean_df['Model'] == models['q4']]['Dynamic_Energy_Joules']
    
    # Get the medians for percentage calculations
    med_fp16 = data_fp16.median()
    med_q8   = data_q8.median()
    med_q4   = data_q4.median()
    
    # Calculate Percentage Savings: ((Baseline - Quantized) / Baseline) * 100
    save_16_to_8 = ((med_fp16 - med_q8) / med_fp16) * 100
    save_16_to_4 = ((med_fp16 - med_q4) / med_fp16) * 100
    save_8_to_4  = ((med_q8 - med_q4) / med_q8) * 100
    
    # Calculate Vargha-Delaney A12 for each quantization step
    u_16_8, _ = stats.mannwhitneyu(data_fp16, data_q8, alternative='two-sided')
    a12_16_8 = u_16_8 / (len(data_fp16) * len(data_q8))
    
    u_16_4, _ = stats.mannwhitneyu(data_fp16, data_q4, alternative='two-sided')
    a12_16_4 = u_16_4 / (len(data_fp16) * len(data_q4))
    
    u_8_4, _ = stats.mannwhitneyu(data_q8, data_q4, alternative='two-sided')
    a12_8_4 = u_8_4 / (len(data_q8) * len(data_q4))
    
    vertical_results.append({
        'Model Family': family_name,
        'FP16 (J)': round(med_fp16, 1),
        'Q8 (J)': round(med_q8, 1),
        'Q4 (J)': round(med_q4, 1),
        'FP16->Q8 Sav.': f"{save_16_to_8:.1f}%",
        'A12 (16->8)': round(a12_16_8, 2),
        'FP16->Q4 Sav.': f"{save_16_to_4:.1f}%",
        'A12 (16->4)': round(a12_16_4, 2),
        'Q8->Q4 Sav.': f"{save_8_to_4:.1f}%",
        'A12 (8->4)': round(a12_8_4, 2)
    })

# Print the vertical table cleanly
es_vertical_df = pd.DataFrame(vertical_results)
print(es_vertical_df.to_string(index=False))

### Energy Delay Product (EDP)

In [ ]:
print("========================================================")
print("============== ENERGY DELAY PRODUCT (EDP) ==============")
print("========================================================\n")

# Calculate the EDP for every single valid run in the dataset
# Formula: EDP = Joules * Seconds
clean_df['EDP_Score'] = clean_df['Dynamic_Energy_Joules'] * clean_df['Total_Time_Seconds']

edp_results = []

# Calculate the Medians for each model
for model in clean_df['Model'].unique():
    if model == 'control': 
        continue
        
    model_df = clean_df[clean_df['Model'] == model]
    
    med_energy = model_df['Dynamic_Energy_Joules'].median()
    med_time = model_df['Total_Time_Seconds'].median()
    med_edp = model_df['EDP_Score'].median()
    
    # Clean up the names for the table
    short_name = model.replace('deepseek-r1_7b-qwen-distill-', 'DS-').replace('qwen2.5_7b-instruct-', 'Qwen-')
    
    edp_results.append({
        'Model (Precision)': short_name.upper(),
        'Median Time (s)': round(med_time, 2),
        'Median Energy (J)': round(med_energy, 1),
        'EDP Score (Lower is Better)': round(med_edp, 1)
    })

# Create a dataframe and sort it so the best model is at the top
edp_df = pd.DataFrame(edp_results)
edp_df = edp_df.sort_values(by='EDP Score (Lower is Better)')

print(edp_df.to_string(index=False))

In [ ]:
# Make sure EDP is calculated for all rows
clean_df['EDP_Score'] = clean_df['Dynamic_Energy_Joules'] * clean_df['Total_Time_Seconds']

clean_df['Short_Name'] = clean_df['Model'].str.replace('deepseek-r1_7b-qwen-distill-', 'DS-Distilled-').str.replace('qwen2.5_7b-instruct-', 'Qwen-')

edp_order = ['Qwen-q4_K_M', 'DS-Distilled-q4_K_M', 'Qwen-q8_0', 'DS-Distilled-q8_0', 'Qwen-fp16', 'DS-Distilled-fp16']

# Set up the visual style
plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")

# Create the Boxplot
ax = sns.boxplot(
    data=clean_df, 
    x='Short_Name', 
    y='EDP_Score', 
    order=edp_order,
    # UPDATED: Orange for Qwen, Blue for DS to match the new order
    palette=['#ff7f0e', '#1f77b4', '#ff7f0e', '#1f77b4', '#ff7f0e', '#1f77b4'], 
    width=0.5,
    fliersize=5, # We KEEP the outlier dots to show volatility!
    linewidth=1.5
)

# Add clear labels and title
plt.title('Energy Delay Product (EDP) Distribution by Model & Precision', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('EDP Score (Joules × Seconds)', fontsize=12, fontweight='bold')
plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=11)

# Format the Y-axis to avoid scientific notation
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

# Save the plot securely without clipping labels
plt.tight_layout()
plt.savefig('../images/experiment-002/edp_boxplot.png', dpi=300)
print("EDP Boxplot successfully saved as 'edp_boxplot.png' in your current folder!")

## EDP for Qwen

In [ ]:
# Make sure EDP is calculated for all rows
clean_df['EDP_Score'] = clean_df['Dynamic_Energy_Joules'] * clean_df['Total_Time_Seconds']

clean_df['Short_Name'] = clean_df['Model'].str.replace('deepseek-r1_7b-qwen-distill-', 'DS-').str.replace('qwen2.5_7b-instruct-', 'Qwen-')

edp_order = ['Qwen-q4_K_M', 'Qwen-q8_0', 'Qwen-fp16']

# Set up the visual style
plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")

# Create the Boxplot
ax = sns.boxplot(
    data=clean_df, 
    x='Short_Name', 
    y='EDP_Score', 
    order=edp_order,
    # UPDATED: Orange for Qwen, Blue for DS to match the new order
    palette=['#ff7f0e','#ff7f0e', '#ff7f0e'], 
    width=0.5,
    fliersize=5, 
    linewidth=1.5
)

# Add clear labels and title
plt.title('Qwen: Energy Delay Product (EDP) Distribution by Precision', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('EDP Score (Joules × Seconds)', fontsize=12, fontweight='bold')
plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=11)

# Format the Y-axis to avoid scientific notation
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

# Save the plot securely without clipping labels
plt.tight_layout()
plt.savefig('../images/experiment-002/edp_boxplot-qwen.png', dpi=300)
print("EDP Boxplot successfully saved as 'edp_boxplot.png' in your current folder!")

## EDP for Deepseek-Distilled Qwen

In [ ]:
# Make sure EDP is calculated for all rows
clean_df['EDP_Score'] = clean_df['Dynamic_Energy_Joules'] * clean_df['Total_Time_Seconds']

clean_df['Short_Name'] = clean_df['Model'].str.replace('deepseek-r1_7b-qwen-distill-', 'DS-Distilled-').str.replace('qwen2.5_7b-instruct-', 'Qwen-')

edp_order = ['DS-Distilled-q4_K_M', 'DS-Distilled-q8_0', 'DS-Distilled-fp16']

# Set up the visual style
plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")

# Create the Boxplot
ax = sns.boxplot(
    data=clean_df, 
    x='Short_Name', 
    y='EDP_Score', 
    order=edp_order,
    # UPDATED: Orange for Qwen, Blue for DS to match the new order
    palette=['#1f77b4','#1f77b4', '#1f77b4'], 
    width=0.5,
    fliersize=5, # We KEEP the outlier dots to show volatility!
    linewidth=1.5
)

# Add clear labels and title
plt.title('DeepSeek: Energy Delay Product (EDP) Distribution by Model', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('EDP Score (Joules × Seconds)', fontsize=12, fontweight='bold')
plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=11)

# Format the Y-axis to avoid scientific notation
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

# Save the plot securely without clipping labels
plt.tight_layout()
plt.savefig('../images/experiment-002/edp_boxplot-deepseek-qwen.png', dpi=300)
print("EDP Boxplot successfully saved as 'edp_boxplot.png' in your current folder!")

### Volatility

In [ ]:
# Sequential Final Summary Table
print("=======================================================")
print("=== FINAL EXECUTIVE SUMMARY: SUSTAINABILITY METRICS ===")
print("=======================================================\n")

final_summary = []

for model in clean_df['Model'].unique():
    if model == 'control': continue
    
    m_df = clean_df[clean_df['Model'] == model]
    
    # Using Median as the representative value for all for consistency
    med_e = m_df['Dynamic_Energy_Joules'].median()
    med_t = m_df['Total_Time_Seconds'].median()
    med_edp = m_df['EDP_Score'].median()
    
    # Calculate Variance (to prove your 'reliability' point)
    # Relative standard deviation (Coefficient of Variation)
    cv_edp = (m_df['EDP_Score'].std() / m_df['EDP_Score'].mean()) * 100

    final_summary.append({
        'Model': model.replace('deepseek-r1_7b-qwen-distill-', 'DS-').replace('qwen2.5_7b-instruct-', 'Qwen-'),
        'Energy (J)': round(med_e, 1),
        'Time (s)': round(med_t, 2),
        'EDP Score': round(med_edp, 1),
        'Volatility (CV%)': f"{cv_edp:.1f}%"
    })

summary_df = pd.DataFrame(final_summary).sort_values('EDP Score')
print(summary_df)

### Temperature - Thermal Efficiency Analysis

In [ ]:
print("===============================================================")
print("========== THERMAL EFFICIENCY: GPU TEMPERATURE DELTA ==========")
print("===============================================================\n")

# Define the exact order we want everything to appear in
desired_order = [
    'Qwen-q4_K_M', 'DS-q4_K_M', 'Qwen-q8_0', 'DS-q8_0', 'Qwen-fp16', 'DS-fp16'
]

# Calculate the Temperature Increase (Max - Start) for every run
clean_df['Temp_Delta'] = clean_df['Max_Temp'] - clean_df['Start_Temp']

thermal_results = []

# Calculate averages and peaks per model
for model in clean_df['Model'].unique():
    if model == 'control': continue
    
    m_df = clean_df[clean_df['Model'] == model]
    
    avg_delta = m_df['Temp_Delta'].mean()
    peak_temp = m_df['Max_Temp'].max()
    
    thermal_results.append({
        'Model': model.replace('deepseek-r1_7b-qwen-distill-', 'DS-').replace('qwen2.5_7b-instruct-', 'Qwen-'),
        'Avg Temp Increase (°C)': round(avg_delta, 2),
        'Peak Temp Reached (°C)': round(peak_temp, 1)
    })

# Convert to DataFrame and enforce the custom sorting order
thermal_df = pd.DataFrame(thermal_results)
thermal_df['Model'] = pd.Categorical(thermal_df['Model'], categories=desired_order, ordered=True)
thermal_df = thermal_df.sort_values('Model')

print(thermal_df.to_string(index=False))
print("\n")

# Create a Correlation Plot: Energy vs. Heat
plt.figure(figsize=(10, 6))

# Clean up names for the legend
plot_df = clean_df[clean_df['Model'] != 'control'].copy()
plot_df['Short_Name'] = plot_df['Model'].str.replace('deepseek-r1_7b-qwen-distill-', 'DS-').str.replace('qwen2.5_7b-instruct-', 'Qwen-')

# Use hue_order to force the legend to match our desired sequence
sns.scatterplot(
    data=plot_df, 
    x='Dynamic_Energy_Joules', 
    y='Temp_Delta', 
    hue='Short_Name', 
    hue_order=desired_order, # Forces the legend order
    alpha=0.7, 
    s=70
)

plt.title('The Cost of Heat: Energy vs. Temperature Increase', fontsize=14, fontweight='bold')
plt.xlabel('Total Energy (Joules)', fontsize=12)
plt.ylabel('Temperature Increase (°C)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

# Move legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Model (Precision)", fontsize='small')

plt.tight_layout()
plt.show()

## Joules per Word

First, we parse the generated responses.txt file. The below cell cleans out all the terminal animation garbage (the spinners and ANSI codes) and counts the exact number of words generated for every single run.
- We calculate the **Median Word Count** for the models.
- We take the **Median Energy (Joules)** and divide it by the **Median Words Count** to find out exactly how much energy it costs to generate one single word.

In [ ]:
print("==============================================================")
print("============== Analysis on Joules per token ===================")
print("==============================================================\n")

# Read the raw responses file
file_path = "../results/experiment-002/responses.txt"
try:
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        raw_text = f.read()
except FileNotFoundError:
    print("Error: 'responses.txt' not found! Make sure the path is correct.")
    raw_text = ""

if raw_text:
    # Clean the terminal garbage (Ollama loading spinners and ANSI codes)
    clean_text = re.sub(r'\[\?[0-9]+[lh]', '', raw_text)
    clean_text = re.sub(r'\[[0-9]+[GK]', '', clean_text)
    clean_text = re.sub(r'[⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏]', '', clean_text)
    clean_text = re.sub(r'\x1b\[[0-9;]*m', '', clean_text) 

    # Parse the file into individual trials and count words
    blocks = re.split(r'===\s+Trial\s+([0-9]+)\s+-\s+Model:\s+([^\s]+)\s+.*?\s+===', clean_text)
    word_counts = []

    for i in range(1, len(blocks), 3):
        model_raw = blocks[i+1]
        text_content = blocks[i+2].strip()
        words = len(text_content.split())
        
        word_counts.append({
            'Model_Raw': model_raw,
            'Word_Count': words
        })

    df_words = pd.DataFrame(word_counts)

    # Merge word counts into our clean_df energy data
    efficiency_results = []
    desired_order = ['Qwen-q4_K_M', 'DS-Distilled-q4_K_M', 'Qwen-q8_0', 'DS-Distilled-q8_0', 'Qwen-fp16', 'DS-Distilled-fp16']

    for model in clean_df['Model'].unique():
        if model == 'control': continue
        
        # Get median energy from clean_df
        m_df = clean_df[clean_df['Model'] == model]
        med_energy = m_df['Dynamic_Energy_Joules'].median()
        
        # Safely match the model name from responses.txt
        search_family = 'qwen2.5' if 'qwen2.5' in model else 'deepseek-r1'
        search_quant = 'q4_K_M' if 'q4' in model else ('q8_0' if 'q8' in model else 'fp16')
        
        match = df_words[(df_words['Model_Raw'].str.contains(search_family)) & 
                         (df_words['Model_Raw'].str.contains(search_quant))]
        
        if not match.empty:
            median_word_count = match['Word_Count'].median()
            joules_per_word = med_energy / median_word_count if median_word_count > 0 else 0
            
            short_name = model.replace('deepseek-r1_7b-qwen-distill-', 'DS-Distilled-').replace('qwen2.5_7b-instruct-', 'Qwen-')
            
            efficiency_results.append({
                'Model': short_name,
                'Median Energy (J)': round(med_energy, 1),
                'Median Words': int(median_word_count),
                'Joules per Word': round(joules_per_word, 2)
            })

    # Display the final Productivity Table
    eff_df = pd.DataFrame(efficiency_results)
    eff_df['Model'] = pd.Categorical(eff_df['Model'], categories=desired_order, ordered=True)
    eff_df = eff_df.sort_values('Model')

    print(eff_df.to_string(index=False))
    print("\n")

    # Plot the Joules per Word
    plt.figure(figsize=(10, 6))
    
    # Keep the colors consistent! Orange for Qwen, Blue for DeepSeek
    ax = sns.barplot(
        data=eff_df, 
        x='Model', 
        y='Joules per Word',
        order=desired_order,
        palette=['#ff7f0e', '#1f77b4', '#ff7f0e', '#1f77b4', '#ff7f0e', '#1f77b4'],
        edgecolor='black',
        alpha=0.8
    )
    
    plt.title('Energy Cost per Word Generated', fontsize=15, fontweight='bold')
    plt.ylabel('Joules per Word', fontsize=12)
    plt.xlabel('Model configuration', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add exact values on top of the bars
    for index, row in enumerate(eff_df.itertuples()):
        plt.text(index, row._4 + 0.05, f"{row._4:.2f} J", color='black', ha="center", fontsize=11, fontweight='bold')
        
    plt.tight_layout()
    plt.savefig('../images/experiment-002/joules_per_word.png', dpi=300) 
    plt.show()

    # -------------------------------------------------------------------------
    # VERBOSITY ANALYSIS
    print("\n==============================================================")
    print("=== VERBOSITY ANALYSIS: QWEN VS DEEPSEEK ====================")
    print("==============================================================\n")

    verbosity_results = []
    precisions = ['q4_K_M', 'q8_0', 'fp16']

    for prec in precisions:
        # Filter the efficiency dataframe for the specific pairs
        qwen_row = eff_df[(eff_df['Model'].str.contains('Qwen')) & (eff_df['Model'].str.contains(prec, case=False))]
        ds_row = eff_df[(eff_df['Model'].str.contains('DS')) & (eff_df['Model'].str.contains(prec, case=False))]
        
        if not qwen_row.empty and not ds_row.empty:
            # Extract the median word counts
            qwen_words = qwen_row['Median Words'].values[0]
            ds_words = ds_row['Median Words'].values[0]
            
            # Calculate the Verbosity Gap: (Qwen - DS) / DS * 100
            extra_words = qwen_words - ds_words
            percent_increase = (extra_words / ds_words) * 100
            
            verbosity_results.append({
                'Precision': prec.upper().replace('_0', '').replace('_K_M', ''),
                'DeepSeek Words': ds_words,
                'Qwen Words': qwen_words,
                'Extra Words Generated': extra_words,
                'Verbosity Gap (% Increase)': f"+{percent_increase:.1f}%"
            })

    verb_df = pd.DataFrame(verbosity_results)
    print(verb_df.to_string(index=False))

    # Calculate the overall average verbosity gap
    avg_gap = verb_df['Verbosity Gap (% Increase)'].str.replace('+', '').str.replace('%', '').astype(float).mean()
    print(f"\n-> OVERALL CONCLUSION: Across all precision levels, the Qwen architecture generates an average of {avg_gap:.1f}% more text per prompt than the distilled DeepSeek architecture.")

## Joules per Token

In [ ]:
print("==============================================================")
print("============== Analysis on Joules per Token ===================")
print("==============================================================\n")

# Read the raw responses file
file_path = "../results/experiment-002/responses.txt"
try:
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        raw_text = f.read()
except FileNotFoundError:
    print("Error: 'responses.txt' not found! Make sure the path is correct.")
    raw_text = ""

if raw_text:
    # Clean the terminal garbage (Ollama loading spinners and ANSI codes)
    clean_text = re.sub(r'\[\?[0-9]+[lh]', '', raw_text)
    clean_text = re.sub(r'\[[0-9]+[GK]', '', clean_text)
    clean_text = re.sub(r'[⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏]', '', clean_text)
    clean_text = re.sub(r'\x1b\[[0-9;]*m', '', clean_text) 

    # Parse the file into individual trials and count words
    blocks = re.split(r'===\s+Trial\s+([0-9]+)\s+-\s+Model:\s+([^\s]+)\s+.*?\s+===', clean_text)
    token_counts = []

    # INDUSTRY STANDARD: 1 Word is approximately 1.33 Tokens
    TOKEN_MULTIPLIER = 1.33

    for i in range(1, len(blocks), 3):
        model_raw = blocks[i+1]
        text_content = blocks[i+2].strip()
        
        # Calculate Estimated Tokens
        words = len(text_content.split())
        estimated_tokens = int(words * TOKEN_MULTIPLIER)
        
        token_counts.append({
            'Model_Raw': model_raw,
            'Token_Count': estimated_tokens
        })

    df_tokens = pd.DataFrame(token_counts)

    # Merge token counts into our clean_df energy data
    efficiency_results = []
    desired_order = ['Qwen-q4_K_M', 'DS-Distilled-q4_K_M', 'Qwen-q8_0', 'DS-Distilled-q8_0', 'Qwen-fp16', 'DS-Distilled-fp16']

    for model in clean_df['Model'].unique():
        if model == 'control': continue
        
        # Get median dynamic energy from clean_df
        m_df = clean_df[clean_df['Model'] == model]
        med_energy = m_df['Dynamic_Energy_Joules'].median()
        
        # Safely match the model name from responses.txt
        search_family = 'qwen2.5' if 'qwen2.5' in model else 'deepseek-r1'
        search_quant = 'q4_K_M' if 'q4' in model else ('q8_0' if 'q8' in model else 'fp16')
        
        match = df_tokens[(df_tokens['Model_Raw'].str.contains(search_family)) & 
                          (df_tokens['Model_Raw'].str.contains(search_quant))]
        
        if not match.empty:
            median_token_count = match['Token_Count'].median()
            # Calculate Joules per Token
            joules_per_token = med_energy / median_token_count if median_token_count > 0 else 0
            
            short_name = model.replace('deepseek-r1_7b-qwen-distill-', 'DS-Distilled-').replace('qwen2.5_7b-instruct-', 'Qwen-')
            
            efficiency_results.append({
                'Model': short_name,
                'Median Dynamic Energy (J)': round(med_energy, 1),
                'Median Tokens Generated': int(median_token_count),
                'Joules per Token': round(joules_per_token, 3) # 3 decimals because it's a smaller number!
            })

    # Display the final Productivity Table
    eff_df = pd.DataFrame(efficiency_results)
    eff_df['Model'] = pd.Categorical(eff_df['Model'], categories=desired_order, ordered=True)
    eff_df = eff_df.sort_values('Model')

    print(eff_df.to_string(index=False))
    print("\n")

    # Plot the Joules per Token
    plt.figure(figsize=(10, 6))
    
    ax = sns.barplot(
        data=eff_df, 
        x='Model', 
        y='Joules per Token',
        order=desired_order,
        palette=['#ff7f0e', '#1f77b4', '#ff7f0e', '#1f77b4', '#ff7f0e', '#1f77b4'],
        edgecolor='black',
        alpha=0.8
    )
    
    plt.title('Energy Cost per Token', fontsize=15, fontweight='bold')
    plt.ylabel('Joules per Token (J/tok)', fontsize=12)
    plt.xlabel('Model Version', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add exact values on top of the bars
    for index, row in enumerate(eff_df.itertuples()):
        plt.text(index, row._4 + (row._4 * 0.02), f"{row._4:.3f} J", color='black', ha="center", fontsize=11, fontweight='bold')
           
    plt.tight_layout()
    plt.savefig('../images/experiment-002/joules_per_token.png', dpi=300)
    plt.show()

    # -------------------------------------------------------------------------
    # VERBOSITY ANALYSIS
    print("\n==============================================================")
    print("=== VERBOSITY ANALYSIS: QWEN VS DEEPSEEK ====================")
    print("==============================================================\n")

    verbosity_results = []
    precisions = ['q4_K_M', 'q8_0', 'fp16']

    for prec in precisions:
        # Filter the efficiency dataframe for the specific pairs
        qwen_row = eff_df[(eff_df['Model'].str.contains('Qwen')) & (eff_df['Model'].str.contains(prec, case=False))]
        ds_row = eff_df[(eff_df['Model'].str.contains('DS')) & (eff_df['Model'].str.contains(prec, case=False))]
        
        if not qwen_row.empty and not ds_row.empty:
            qwen_tokens = qwen_row['Median Tokens Generated'].values[0]
            ds_tokens = ds_row['Median Tokens Generated'].values[0]
            
            # Calculate the Verbosity Gap
            extra_tokens = qwen_tokens - ds_tokens
            percent_increase = (extra_tokens / ds_tokens) * 100
            
            verbosity_results.append({
                'Precision': prec.upper().replace('_0', '').replace('_K_M', ''),
                'DeepSeek Tokens': ds_tokens,
                'Qwen Tokens': qwen_tokens,
                'Extra Tokens Generated': extra_tokens,
                'Verbosity Gap (% Increase)': f"+{percent_increase:.1f}%"
            })

    verb_df = pd.DataFrame(verbosity_results)
    print(verb_df.to_string(index=False))

    avg_gap = verb_df['Verbosity Gap (% Increase)'].str.replace('+', '').str.replace('%', '').astype(float).mean()
    print(f"\n-> OVERALL CONCLUSION: Across all precision levels, the Qwen architecture generates an average of {avg_gap:.1f}% more tokens per prompt than the distilled DeepSeek architecture.")